# 12. Spectrum Forecasting via SSA–LSTM–GWO (Decomposition–Prediction–Optimization)

Implements the **Decomposition–Prediction–Optimization** hybrid framework from the paper:
**"Time Series Forecasting Study Based on Decomposition-Prediction-Optimization Mode"** (ICCECT 2024).

## Paper pipeline adapted to your use case
- **Decomposition (SSA):** Decompose the AU% time series into multiple SSA components.
- **Prediction (LSTM):** Train an LSTM to predict each component.
- **Optimization (GWO):** Use Grey Wolf Optimization (GWO) to learn **weights** for combining component predictions into the final next-day forecast.

## Your task
- **Input:** last **72 hours** (3 days)
- **Output:** next **24 hours** (next day)
- **Data source:** `work_dir/final/training/` and `work_dir/final/testing/` (same as notebook 10)
- **Metrics:** MAE, RMSE, MASE
- **Visuals:** Predicted vs Actual (dashed), mean profile, per-hour MAE, residuals, improvement vs baseline


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

print(f"TensorFlow: {tf.__version__}")
print(f"CPU cores: {os.cpu_count()}")

In [ ]:
# GPU/Metal setup for Mac (same idea as notebook 10)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus, 'GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU enabled: {len(gpus)} device(s): {[g.name for g in gpus]}")
        with tf.device('/GPU:0'):
            _ = tf.constant(1)
        print("GPU ready.")
    except RuntimeError as e:
        print("GPU config:", e)
else:
    print("No GPU detected. On Apple Silicon install: pip install tensorflow-metal")

USE_GPU = len(gpus) > 0

# Use all CPU cores for input pipeline + CPU ops
num_cores = os.cpu_count() or 4
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)
print(f"CPU threads: intra={num_cores}, inter={num_cores}")

## Data loading (same as notebook 10)

In [ ]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
if not final_dir.exists():
    raise FileNotFoundError("Final dir not found: " + str(final_dir))

training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError("Training or testing directory not found")

class_options = sorted([d.name for d in training_dir.iterdir() if d.is_dir()])
print(f"Bands: {class_options}")

LOOKBACK = 72
FORECAST_HORIZON = 24
N_TEST_DAYS = 3
print(f"Lookback={LOOKBACK}h, horizon={FORECAST_HORIZON}h")

def load_data_for_band(band_name: str, split: str):
    split_dir = final_dir / split / band_name
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        dfs.append(pd.read_parquet(p))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def get_series_per_freq(train_df: pd.DataFrame, test_df: pd.DataFrame):
    # Use first threshold if multiple
    ths = sorted(train_df["threshold_dbm"].unique())
    if len(ths) > 1:
        train_df = train_df[train_df["threshold_dbm"] == ths[0]].copy()
        test_df = test_df[test_df["threshold_dbm"] == ths[0]].copy()
    out = {}
    for freq in sorted(train_df["freq_center_ghz"].unique()):
        tr = train_df[train_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        te = test_df[test_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        if len(tr) and len(te):
            out[freq] = (tr.astype(float), te.astype(float))
    return out

train_data_by_band = {b: load_data_for_band(b, "training") for b in class_options}
test_data_by_band = {b: load_data_for_band(b, "testing") for b in class_options}
print("Loaded bands:", [b for b in class_options if not train_data_by_band[b].empty])

## SSA decomposition into components

Paper uses SSA decomposition; here we reconstruct the first `N_COMP` elementary components (each from one singular triplet).

In [ ]:
def _diag_average(X: np.ndarray) -> np.ndarray:
    # anti-diagonal averaging for LxK matrix
    L, K = X.shape
    n = L + K - 1
    y = np.zeros(n)
    for k in range(n):
        i_lo = max(0, k - K + 1)
        i_hi = min(k, L - 1)
        y[k] = np.mean([X[i, k - i] for i in range(i_lo, i_hi + 1)])
    return y

def ssa_decompose(series: np.ndarray, L: int, n_comp: int = 5):
    """Return list of SSA components (length n_comp) and a residual component."""
    x = series.astype(float)
    n = len(x)
    if L < 2 or n < L:
        return [x.copy()], np.zeros_like(x)
    K = n - L + 1
    X = np.column_stack([x[i:i+L] for i in range(K)])
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    m = min(n_comp, len(s))
    comps = []
    for i in range(m):
        Xi = (U[:, [i]] * s[i]) @ Vt[[i], :]
        comps.append(_diag_average(Xi))
    recon = np.sum(comps, axis=0)
    resid = x - recon
    return comps, resid

# Hyperparams from paper example (SSA embedding window length = 20)
L_SSA = 20
N_COMP = 5
print(f"SSA: L={L_SSA}, components={N_COMP}")

## LSTM predictor (per component)

We predict each SSA component separately using an LSTM, then combine components with GWO-optimized weights.

In [ ]:
def make_supervised(series: np.ndarray, lookback: int, horizon: int):
    X, y = [], []
    for i in range(len(series) - lookback - horizon + 1):
        X.append(series[i:i+lookback])
        y.append(series[i+lookback:i+lookback+horizon])
    X = np.array(X).reshape(-1, lookback, 1)
    y = np.array(y)
    return X, y

def build_lstm_model(lookback: int, horizon: int):
    model = keras.Sequential([
        layers.Input(shape=(lookback, 1)),
        layers.LSTM(32, activation='tanh'),
        layers.Dense(horizon, activation='linear'),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

EPOCHS = 50
BATCH_SIZE = 128 if USE_GPU else 32
print(f"Training params: epochs={EPOCHS}, batch={BATCH_SIZE}")

## GWO (Grey Wolf Optimizer) for combining component predictions

Paper uses GWO to weight the reconstructed prediction. We optimize weights on a validation slice of the training set.


In [ ]:
def gwo_optimize(objective_fn, dim: int, lb: float, ub: float, pop: int = 20, iters: int = 60, seed: int = 0):
    rng = np.random.default_rng(seed)
    X = rng.uniform(lb, ub, size=(pop, dim))
    # alpha, beta, delta
    scores = np.array([objective_fn(x) for x in X])
    order = np.argsort(scores)
    alpha, beta, delta = X[order[0]].copy(), X[order[1]].copy(), X[order[2]].copy()
    alpha_s = scores[order[0]]
    for t in range(iters):
        a = 2 - 2 * (t / max(1, iters - 1))
        for i in range(pop):
            for leader in (alpha, beta, delta):
                r1, r2 = rng.random(dim), rng.random(dim)
                A = 2 * a * r1 - a
                C = 2 * r2
                D = np.abs(C * leader - X[i])
                X1 = leader - A * D
                if leader is alpha:
                    acc = X1
                else:
                    acc = acc + X1
            X[i] = np.clip(acc / 3.0, lb, ub)
        scores = np.array([objective_fn(x) for x in X])
        order = np.argsort(scores)
        alpha, beta, delta = X[order[0]].copy(), X[order[1]].copy(), X[order[2]].copy()
        alpha_s = scores[order[0]]
    return alpha, alpha_s

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


## Train SSA–LSTM per component, then GWO weights, then evaluate

We evaluate on **trend+components reconstruction** (combined prediction) and report MAE/RMSE/MASE.


In [ ]:
def mase(y_true: np.ndarray, y_pred: np.ndarray, y_train: np.ndarray) -> float:
    mae_val = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    scale = np.mean(np.abs(np.diff(y_train.flatten()))) if len(y_train.flatten()) > 1 else 1.0
    if scale == 0:
        scale = 1.0
    return float(mae_val / scale)

def naive_last_value(y_hist: np.ndarray, horizon: int) -> np.ndarray:
    return np.full(horizon, float(y_hist[-1]))

# Collect test day samples (one per day per freq per band) just like notebook 10
y_true_all = []
y_pred_all = []
y_pred_naive_all = []
y_train_all = []

# To keep runtime reasonable: train one set of component LSTMs per (band,freq)
for band in class_options:
    tr_df = train_data_by_band.get(band)
    te_df = test_data_by_band.get(band)
    if tr_df is None or te_df is None or tr_df.empty or te_df.empty:
        continue
    series_map = get_series_per_freq(tr_df, te_df)
    for freq, (tr, te) in series_map.items():
        full = np.concatenate([tr, te])
        comps, resid = ssa_decompose(full, L=L_SSA, n_comp=N_COMP)
        # include residual as an extra component (optional)
        comp_list = comps + [resid]
        comp_dim = len(comp_list)

        n_train = len(tr)
        # Build component training series and a small validation slice (last 24h of training)
        comp_train = [c[:n_train] for c in comp_list]
        comp_test = [c[n_train:] for c in comp_list]  # length 72

        # Train per-component LSTMs on training portion
        comp_models = []
        comp_scalers = []
        # create a validation target for weighting: last day of training (t-24..t-1) -> predict next day (t..t+23)
        # We'll use the final supervised sample inside training to validate weights.
        val_true = None
        val_preds_components = []

        for c in comp_train:
            # scale per component for stable LSTM training
            sc = MinMaxScaler(feature_range=(0, 1))
            c_s = sc.fit_transform(c.reshape(-1, 1)).ravel()
            Xc, yc = make_supervised(c_s, LOOKBACK, FORECAST_HORIZON)
            if len(Xc) < 2:
                comp_models.append(None)
                comp_scalers.append(sc)
                continue
            # last sample as validation for GWO
            X_train_c, y_train_c = Xc[:-1], yc[:-1]
            X_val_c, y_val_c = Xc[-1:], yc[-1:]
            if val_true is None:
                val_true = y_val_c.copy()
            model = build_lstm_model(LOOKBACK, FORECAST_HORIZON)
            model.fit(X_train_c, y_train_c, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
            val_pred_c = model.predict(X_val_c, verbose=0)[0]
            val_preds_components.append(val_pred_c)
            comp_models.append(model)
            comp_scalers.append(sc)

        if val_true is None or len(val_preds_components) == 0:
            continue

        val_preds_components = np.array(val_preds_components)  # (comp_dim_eff, 24)
        val_true_vec = val_true[0]  # (24,)

        # Objective: minimize RMSE of weighted sum on validation
        def obj(w):
            # w shape (comp_dim_eff,) — allow any weights, then normalize
            w = np.array(w, dtype=float)
            w = w / (np.sum(np.abs(w)) + 1e-8)
            pred = (w.reshape(-1, 1) * val_preds_components).sum(axis=0)
            return rmse(val_true_vec, pred)

        w_best, _ = gwo_optimize(obj, dim=val_preds_components.shape[0], lb=0.95, ub=1.05, pop=20, iters=60, seed=0)
        w_best = w_best / (np.sum(np.abs(w_best)) + 1e-8)

        # Predict the 3 test days (72 steps) component-wise using the trained models
        # Use iterative day-by-day prediction using model outputs.
        hist_components = [c[:n_train].copy() for c in comp_list]
        for day in range(N_TEST_DAYS):
            pred_components_day = []
            for model, sc, hist in zip(comp_models, comp_scalers, hist_components):
                if model is None:
                    pred_components_day.append(np.zeros(FORECAST_HORIZON))
                    continue
                # prepare last LOOKBACK in scaled space
                last = hist[-LOOKBACK:]
                last_s = sc.transform(last.reshape(-1, 1)).ravel()
                X_in = last_s.reshape(1, LOOKBACK, 1)
                y_pred_s = model.predict(X_in, verbose=0)[0]
                y_pred = sc.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
                pred_components_day.append(y_pred)
            pred_components_day = np.array(pred_components_day)  # (comp_dim_eff, 24)
            y_pred_day = (w_best.reshape(-1, 1) * pred_components_day).sum(axis=0)

            # ground truth is sum of true components for that test day
            start = day * FORECAST_HORIZON
            end = (day + 1) * FORECAST_HORIZON
            y_true_day = np.sum([ct[start:end] for ct in comp_test], axis=0)

            y_true_all.append(y_true_day)
            y_pred_all.append(y_pred_day)

            # naive baseline on reconstructed series
            recon_hist = np.sum([h for h in hist_components], axis=0)
            y_pred_naive_all.append(naive_last_value(recon_hist, FORECAST_HORIZON))

            # extend histories with *predicted* components to continue multi-day
            for idx in range(len(hist_components)):
                hist_components[idx] = np.concatenate([hist_components[idx], pred_components_day[idx]])

        # for MASE scaling use reconstructed training series
        y_train_all.append(np.sum([c[:n_train] for c in comp_list], axis=0))

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)
y_pred_naive_all = np.array(y_pred_naive_all)
y_train_all = np.concatenate(y_train_all) if len(y_train_all) else np.array([])

print("Shapes:")
print(" y_true:", y_true_all.shape)
print(" y_pred:", y_pred_all.shape)
print(" y_naive:", y_pred_naive_all.shape)

In [ ]:
mae = mean_absolute_error(y_true_all.flatten(), y_pred_all.flatten())
rmse_val = np.sqrt(mean_squared_error(y_true_all.flatten(), y_pred_all.flatten()))
mase_val = mase(y_true_all, y_pred_all, y_train_all)

mae_n = mean_absolute_error(y_true_all.flatten(), y_pred_naive_all.flatten())
rmse_n = np.sqrt(mean_squared_error(y_true_all.flatten(), y_pred_naive_all.flatten()))
mase_n = mase(y_true_all, y_pred_naive_all, y_train_all)

results_df = pd.DataFrame([
    {"Model": "SSA-LSTM-GWO", "MAE": mae, "RMSE": rmse_val, "MASE": mase_val},
    {"Model": "Naive Baseline", "MAE": mae_n, "RMSE": rmse_n, "MASE": mase_n},
])
results_df['MAE'] = results_df['MAE'].round(4)
results_df['RMSE'] = results_df['RMSE'].round(4)
results_df['MASE'] = results_df['MASE'].round(4)

print('\n' + '='*80)
print('FINAL RESULTS SUMMARY (same format as notebook 10 and 11)')
print('='*80)
print(f'Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h')
print(f'Test samples: {len(y_true_all)}')
print(f'\n{results_df.to_string(index=False)}')
display(results_df)

## Visuals (similar to notebook 10)

In [ ]:
hours = np.arange(FORECAST_HORIZON)
n_show = min(3, len(y_true_all))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1:
    axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_true_all[i], 'k--', linewidth=2.5, label='Actual', alpha=0.8)
    ax.plot(hours, y_pred_all[i], '-', linewidth=1.6, label='SSA-LSTM-GWO')
    ax.plot(hours, y_pred_naive_all[i], '-', linewidth=1.2, label='Naive', alpha=0.7)
    ax.set_title(f"Test sample {i+1}: Predicted (solid) vs Actual (dashed)")
    ax.set_xlabel("Hour")
    ax.set_ylabel("AU (%)")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

# Mean 24h profile
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_true_all.mean(axis=0), 'k--', linewidth=2.5, label='Actual (mean)')
ax.plot(hours, y_pred_all.mean(axis=0), '-', linewidth=1.6, label='SSA-LSTM-GWO (mean)')
ax.plot(hours, y_pred_naive_all.mean(axis=0), '-', linewidth=1.2, label='Naive (mean)', alpha=0.7)
ax.set_title('Mean 24h profile: predicted vs actual')
ax.set_xlabel('Hour')
ax.set_ylabel('AU (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-hour MAE
mae_per_hour = np.abs(y_true_all - y_pred_all).mean(axis=0)
mae_per_hour_naive = np.abs(y_true_all - y_pred_naive_all).mean(axis=0)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, mae_per_hour, '-o', label='SSA-LSTM-GWO', markersize=4)
ax.plot(hours, mae_per_hour_naive, '-o', label='Naive', markersize=4, alpha=0.7)
ax.set_title('MAE by forecast hour')
ax.set_xlabel('Hour')
ax.set_ylabel('MAE (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 1. Bar charts: MAE, RMSE, MASE by model (same as notebook 10)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MASE']
colors = ['#2ecc71', '#3498db', '#9b59b6']
for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].values
    bars = ax.bar(results_df['Model'], vals, color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=15)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02 * max(vals), f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.suptitle('Next-day prediction: metric comparison', y=1.02, fontsize=12)
plt.show()

In [ ]:
# 2. Improvement over Naive Baseline (%) — same format as notebook 10
naive_mae = results_df[results_df['Model'] == 'Naive Baseline']['MAE'].values[0]
naive_rmse = results_df[results_df['Model'] == 'Naive Baseline']['RMSE'].values[0]
naive_mase = results_df[results_df['Model'] == 'Naive Baseline']['MASE'].values[0]
improvement = results_df[results_df['Model'] != 'Naive Baseline'].copy()
improvement['MAE_imp_%'] = (1 - improvement['MAE'] / naive_mae) * 100
improvement['RMSE_imp_%'] = (1 - improvement['RMSE'] / naive_rmse) * 100
improvement['MASE_imp_%'] = (1 - improvement['MASE'] / naive_mase) * 100
print('Improvement over Naive Baseline (%):')
display(improvement[['Model', 'MAE_imp_%', 'RMSE_imp_%', 'MASE_imp_%']].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(improvement))
w = 0.25
ax.bar(x - w, improvement['MAE_imp_%'], w, label='MAE', color='#2ecc71')
ax.bar(x, improvement['RMSE_imp_%'], w, label='RMSE', color='#3498db')
ax.bar(x + w, improvement['MASE_imp_%'], w, label='MASE', color='#9b59b6')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(improvement['Model'], rotation=15)
ax.set_ylabel('Improvement (%)')
ax.legend()
ax.set_title('Improvement over Naive Baseline (positive = better)')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Residuals: SSA-LSTM-GWO vs Naive (same as notebook 10)
residuals_best = (y_true_all - y_pred_all).flatten()
residuals_naive = (y_true_all - y_pred_naive_all).flatten()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals_best, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual (actual - predicted)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residuals: SSA-LSTM-GWO')
axes[1].hist(residuals_naive, bins=50, color='#95a5a6', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (actual - predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals: Naive Baseline')
plt.suptitle('Residual distribution (centered at 0 is ideal)', y=1.02)
plt.tight_layout()
plt.show()
print(f'Residual mean (bias): SSA-LSTM-GWO = {residuals_best.mean():.4f}, Naive = {residuals_naive.mean():.4f}')
print(f'Residual std:         SSA-LSTM-GWO = {residuals_best.std():.4f}, Naive = {residuals_naive.std():.4f}')

### Key insights

- **Best model:** SSA-LSTM-GWO vs Naive: check the results table and bar charts; the model with lowest MAE/RMSE/MASE is better for next-day AU prediction.
- **Improvement over naive:** Positive % in the improvement plot means SSA-LSTM-GWO beats the naive (last-value) baseline.
- **Mean profile:** If predicted and actual mean profiles align, the model captures the typical daily pattern; gaps indicate systematic bias.
- **Per-hour MAE:** Peaks at certain hours suggest harder-to-predict periods.
- **Residuals:** A symmetric, centered distribution indicates unbiased predictions; large spread means higher uncertainty.

In [ ]:
# One-line summary (same format as notebook 10)
best_dl = 'SSA-LSTM-GWO'
best_row = results_df[results_df['Model'] == best_dl].iloc[0]
imp_mae = (1 - best_row['MAE'] / naive_mae) * 100
print(f'Best model: {best_dl} (MAE={best_row["MAE"]:.4f}, RMSE={best_row["RMSE"]:.4f}, MASE={best_row["MASE"]:.4f}).')
print(f'Improvement over Naive: MAE {imp_mae:+.1f}%.')